# Frequentist & Bayesian Analysis

In [ ]:
import os, sys
from pathlib import Path
_cwd = Path.cwd()
_root = next((p for p in [_cwd] + list(_cwd.parents)
              if (p / 'requirements.txt').exists() and (p / 'src').is_dir()), None)
assert _root, f'Could not find project root from {_cwd}'
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f'Project root: {_root}')


## Frequentist & Bayesian Analysis — 7-day Retention

We run both a **two-proportion z-test** and a **Beta-Binomial Bayesian** analysis on the
Cookie Cats 7-day retention metric.

A **bootstrap CI** (2 000 resamples) serves as an external cross-check on the parametric CI.

> If `cookie_cats.csv` is absent, we fall back to a **simulated dataset** at the same scale
> and note this clearly below.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from src.abtest import (
    ExperimentData, simulate_conversion,
    two_proportion_z, beta_binomial,
)

csv = Path('data/raw/cookie_cats.csv')
if csv.exists():
    df = pd.read_csv(csv)
    df2 = df.rename(columns={'version': 'variant', 'retention_7': 'converted'})
    df2['variant'] = df2['variant'].map({'gate_30': 'control', 'gate_40': 'treatment'})
    data = ExperimentData(
        df2[['variant', 'converted']].assign(unit_id=range(len(df2))),
        metric_cols=['converted'],
    )
    using_sim = False
    print("Using real Cookie Cats data.")
else:
    print("cookie_cats.csv not found — using SIMULATED data as stand-in.")
    print("Parameters: n_per_arm=45000, base_rate=0.19, absolute_lift=-0.008, seed=7")
    sim = simulate_conversion(n_per_arm=45_000, base_rate=0.19, absolute_lift=-0.008, seed=7)
    data = sim
    using_sim = True


### Frequentist: Two-proportion z-test


In [ ]:
freq = two_proportion_z(data, 'converted', alpha=0.05)
print(f"Control rate     : {freq.control_mean:.4f}")
print(f"Treatment rate   : {freq.treatment_mean:.4f}")
print(f"Absolute effect  : {freq.absolute_effect:+.4f}")
print(f"Relative effect  : {freq.relative_effect:+.2%}")
print(f"95% CI           : [{freq.ci_low:+.4f}, {freq.ci_high:+.4f}]")
print(f"p-value          : {freq.p_value:.4f}")
print(f"Significant      : {freq.significant}")
print(f"Verdict          : {freq.verdict}")


### Bayesian: Beta-Binomial


In [ ]:
bayes = beta_binomial(data, 'converted', seed=42)
print(f"P(treatment better)       : {bayes.prob_treatment_better:.4f}")
print(f"Expected loss (treatment) : {bayes.expected_loss_treatment:.6f}")
print(f"Expected loss (control)   : {bayes.expected_loss_control:.6f}")
print(f"95% credible interval     : [{bayes.cred_low:+.4f}, {bayes.cred_high:+.4f}]")
print(f"Verdict                   : {bayes.verdict}")


### Bootstrap CI (external cross-check)


In [ ]:
rng = np.random.default_rng(0)
ctrl = data.control['converted'].values.astype(float)
trt  = data.treatment['converted'].values.astype(float)

diffs = []
for _ in range(2_000):
    bc = rng.choice(ctrl, size=len(ctrl), replace=True)
    bt = rng.choice(trt,  size=len(trt),  replace=True)
    diffs.append(bt.mean() - bc.mean())

diffs = np.array(diffs)
boot_lo, boot_hi = np.percentile(diffs, [2.5, 97.5])
print(f"Bootstrap 95% CI for difference: [{boot_lo:+.4f}, {boot_hi:+.4f}]")
print(f"Bootstrap point estimate       : {diffs.mean():+.4f}")


### Conclusion

The published Cookie Cats finding is that the **7-day retention difference is NOT statistically
significant** — the gate move from 30 to 40 had little detectable impact on 7-day retention at
conventional thresholds.

Both the frequentist p-value and Bayesian posterior should reflect this: the 95% CI
straddles zero and P(treatment better) is close to 0.5.

> Note: if using simulated data (no CSV found), the simulated lift of −0.008 mimics the
> direction seen in the original data. Results will be similar in magnitude but not identical.
